In [1]:
!pip install razdel pymorphy3 spacy nltk wordcloud pandas matplotlib regex
!python -m spacy download ru_core_news_sm

urllib3.exceptions.SSLError: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1082)

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Anaconda\Lib\site-packages\requests\adapters.py", line 696, in send
    resp = conn.urlopen(
        method=request.method,
    ...<9 lines>...
        chunked=chunked,
    )
  File "C:\Anaconda\Lib\site-packages\urllib3\connectionpool.py", line 842, in urlopen
    retries = retries.increment(
        method, url, error=new_e, _pool=self, _stacktrace=sys.exc_info()[2]
    )
  File "C:\Anaconda\Lib\site-packages\urllib3\util\retry.py", line 543, in increment
    raise MaxRetryError(_pool, url, reason) from reason  # type: ignore[arg-type]
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
urllib3.exceptions.MaxRetryError: HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /explosion/spacy-models/master/compa

In [2]:
import os

import regex as re
from collections import Counter

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

from razdel import tokenize
import pymorphy3

morph = pymorphy3.MorphAnalyzer()
stop_words = set(stopwords.words("russian"))

import spacy
nlp = spacy.load("ru_core_news_sm")

In [3]:
# Текущая директория
print("Текущая папка:", os.getcwd())

# Содержимое папки data/
data_dir = "data"
print("\nСодержимое data/:", os.listdir(data_dir)[:10], "...")

# Проверка существования файла
file_path = os.path.join(data_dir, "erdman.txt")
print("\nФайл существует:", os.path.exists(file_path))

# Разделение имени и расширения
name, ext = os.path.splitext(file_path)
print("\nИмя:", name)
print("Расширение:", ext)

Текущая папка: C:\Users\User\Downloads\образцы кодов

Содержимое data/: ['.DS_Store', '.ipynb_checkpoints', 'alice_in_wonderland_ch1.txt', 'data', 'federal_assembly_address_1994.txt', 'federal_assembly_address_1998.txt', 'federal_assembly_address_2002.txt', 'federal_assembly_address_2006.txt', 'federal_assembly_address_2010.txt', 'federal_assembly_address_2014.txt'] ...

Файл существует: False

Имя: data\erdman
Расширение: .txt


In [4]:
erdman_path = os.path.join("erdman.txt")

with open(erdman_path, "r", encoding="utf-8") as f:
    erdman_text = f.read()
#print(erdman_text[:1000]) - проверяю выдачу

In [5]:
clean_text = re.sub(r'\[\d+\]', ' ', erdman_text)
clean_text = re.sub(r'[ \t]+', ' ', clean_text)
clean_text = re.sub(r'\n{2,}', '\n', clean_text)


In [6]:
blocks = clean_text.split('* * *')

memories = []
letters = []

for block in blocks:
    if '[Воспоминания А. И. Степановой]' in block:
        memories.append(block)
    else:
        letters.append(block)

len(memories)

7

In [7]:
only_letters = '* * *'.join(letters)
letters_blocks = re.split(r'\s*\*\s*\*\s*\*\s*', only_letters)
letters_blocks = [block.strip() for block in letters_blocks if block.strip()]

In [8]:
month_pattern = r'(?:января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря|янв\.|февр?\.|мар\.|апр\.|июн\.|июл\.|авг\.|сент\.|окт\.|нояб?\.|дек\.|январь|февраль|март|апрель|май|июнь|июль|август|сентябрь|октябрь|ноябрь|декабрь)'

date_pattern = r'\[?\s*(?:\d{1,2}[./]\d{1,2}[./]\d{2,4}|\d{1,2}\s+' + month_pattern + r'(?:\s+\d{2,4}\s*(?:г\.?|года)?)?|' + month_pattern + r')\s*\]?'

dates = []

for block in letters_blocks:
    header = '\n'.join(block.splitlines()[:10])

    match = re.search(date_pattern, header, re.IGNORECASE)

    if match:
        dates.append(match.group().strip(' []'))
    else:
        dates.append(None)

from collections import Counter

print(Counter(dates)) #проверяю выдачу

Counter({None: 13, '21 августа\n': 2, '23 августа\n': 2, '24 августа\n': 2, '25 августа\n': 2, '\n3 ноября 1933 г.\n': 1, '4 ноября 1933 г': 1, '5 ноября\n': 1, '6 ноября\n': 1, '\n20 ноября 1933 г.\n': 1, '22 ноября 1933 г.\n': 1, '22 ноября\n': 1, '23 ноября\n': 1, '24 ноября\n': 1, '\n3 декабря 1933 г.\n': 1, '\n12 декабря 33 г.\n': 1, '12 декабря 33 г.\n': 1, '12 декабря\n': 1, '\n19 дек. 33 г.\n': 1, '\n26 дек. 33 г.\n': 1, '1 января 34 г.': 1, '\n10 янв. 34 г.\n': 1, '11 января\n': 1, '20.01.34': 1, '\n19 февр. 34 г.\n': 1, 'декабрь': 1, '\n24 марта 34 г.\n': 1, '27 марта\n': 1, '2.04.34': 1, '10 апреля\n': 1, '11 апреля\n': 1, '17.04.34': 1, '25 апреля 34 г.': 1, '1 мая\n': 1, 'июня': 1, '1 июня\n': 1, '2 июня\n': 1, '19 августа\n': 1, '20 августа\n': 1, '22 августа\n': 1, '26 августа\n': 1, '3 сентября\n': 1, '5.09.34': 1, '5 сентября\n': 1, 'Ноябрь\n': 1, '\n6 нояб. 34 г.\n': 1, '21 ноября\n': 1, '\n14.12.34': 1, '22 февраля\n': 1, '7.03.35\n': 1, '\n20.03.35': 1, '1.04.35': 1

In [9]:
authors = []

for block in letters_blocks:
    author = None

    if re.search(r'Николай\.?\s*$', block):
        author = 'Эрдман'

    elif re.search(r'Лина\.?\s*$', block):
        author = 'Степанова'

    else:
        header = block[:500]

        if re.search(r'Ангелине Осиповне Степановой|А\. О\. Степановой', header):
            author = 'Эрдман'

        elif re.search(r'Н\. Р\. Эрдман', header):
            author = 'Степанова'

    authors.append(author)

from collections import Counter

print(Counter(authors)) #проверяю выдачу

Counter({'Степанова': 29, 'Эрдман': 27, None: 12})


In [10]:
import pandas as pd
df = pd.DataFrame({
    'дата': dates,
    'автор': authors, 
    'текст письма': letters_blocks
})
df.head()

,дата,автор,текст письма
0,NaN,Эрдман,"Баку,\nОперный театр. Гастроли М.Х.Т.,\nА. О. ..."
1,\n3 ноября 1933 г.\n,Эрдман,"Москва,\nпроезд Художественного театра,\nХудож..."
2,4 ноября 1933 г,Степанова,"4 ноября 1933 года\nДорогой, прекрасный, любим..."
3,5 ноября\n,Степанова,"5 ноября\nДорогой мой, посылка халтурно состав..."
4,6 ноября\n,Степанова,"6 ноября\nНиколашенька, вчера пришла телеграмм..."


In [11]:
all_lemmas = []

for block in letters_blocks:
    words = list(tokenize(block))
    lemmas = []

    for word in words:
        if word.text.isalpha():
            lemma = morph.parse(word.text)[0].normal_form
            lemmas.append(lemma)

    all_lemmas.append(' '.join(lemmas))

df['лемматизированный текст письма'] = all_lemmas
df #проверяю выдачу

,дата,автор,текст письма,лемматизированный текст письма
0,NaN,Эрдман,"Баку,\nОперный театр. Гастроли М.Х.Т.,\nА. О. ...",баку оперный театр гастроль м х так а о степан...
1,\n3 ноября 1933 г.\n,Эрдман,"Москва,\nпроезд Художественного театра,\nХудож...",москва проезд художественный театр художествен...
2,4 ноября 1933 г,Степанова,"4 ноября 1933 года\nДорогой, прекрасный, любим...",ноябрь год дорога прекрасный любимый я хотетьс...
3,5 ноября\n,Степанова,"5 ноября\nДорогой мой, посылка халтурно состав...",ноябрь дорога мой посылка халтурно составить н...
4,6 ноября\n,Степанова,"6 ноября\nНиколашенька, вчера пришла телеграмм...",ноябрь николашенек вчера прийти телеграмма что...
...,...,...,...,...
63,22 февраля\n,Степанова,22 февраля\nСегодня получила в театре твою том...,февраль сегодня получить в театр твой томский ...
64,7.03.35\n,NaN,7.03.35\nНа этот раз хочу заступиться за телег...,на этот раз хотеть заступиться за телеграф лиш...
65,\n20.03.35,Эрдман,"Москва,\nпроезд Художественного театра.\nХудож...",москва проезд художественный театр художествен...
66,1.04.35,NaN,1.04.35 г.\nОпять несколько дней без Твоих отк...,г опять несколько день без твой открытка хотет...


In [12]:
words_count = []

for block in letters_blocks:
    words = list(tokenize(block))
    count = 0
    
    for word in words:
        if word.text.isalpha():
            count +=1
    words_count.append(count)

df['количество слов в письме'] = words_count
df #проверяю выдачу

,дата,автор,текст письма,лемматизированный текст письма,количество слов в письме
0,NaN,Эрдман,"Баку,\nОперный театр. Гастроли М.Х.Т.,\nА. О. ...",баку оперный театр гастроль м х так а о степан...,216
1,\n3 ноября 1933 г.\n,Эрдман,"Москва,\nпроезд Художественного театра,\nХудож...",москва проезд художественный театр художествен...,126
2,4 ноября 1933 г,Степанова,"4 ноября 1933 года\nДорогой, прекрасный, любим...",ноябрь год дорога прекрасный любимый я хотетьс...,124
3,5 ноября\n,Степанова,"5 ноября\nДорогой мой, посылка халтурно состав...",ноябрь дорога мой посылка халтурно составить н...,29
4,6 ноября\n,Степанова,"6 ноября\nНиколашенька, вчера пришла телеграмм...",ноябрь николашенек вчера прийти телеграмма что...,132
...,...,...,...,...,...
63,22 февраля\n,Степанова,22 февраля\nСегодня получила в театре твою том...,февраль сегодня получить в театр твой томский ...,124
64,7.03.35\n,NaN,7.03.35\nНа этот раз хочу заступиться за телег...,на этот раз хотеть заступиться за телеграф лиш...,202
65,\n20.03.35,Эрдман,"Москва,\nпроезд Художественного театра.\nХудож...",москва проезд художественный театр художествен...,190
66,1.04.35,NaN,1.04.35 г.\nОпять несколько дней без Твоих отк...,г опять несколько день без твой открытка хотет...,300


In [13]:
df.info()
df.value_counts('автор') 

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 5 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   дата                            55 non-null     str  
 1   автор                           56 non-null     str  
 2   текст письма                    68 non-null     str  
 3   лемматизированный текст письма  68 non-null     str  
 4   количество слов в письме        68 non-null     int64
dtypes: int64(1), str(4)
memory usage: 243.6 KB


автор
Степанова    29
Эрдман       27
Name: count, dtype: int64

In [14]:
df.to_csv('correspondence.csv')